# Task 5: Exploratory Analysis

**Goal:** Open-ended exploration. Suggested directions:
- Compare tumor vs adjacent normal ERBB2 expression
- Differential expression in HER2+ vs HER2−
- Triple-negative breast cancer (TNBC) subtyping
- Clinical correlates of HER2 positivity (age, stage, grade)
- Pathway enrichment analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_utils import load_processed
from src.plotting import save_fig

In [ ]:
rna = load_processed('rna_normalized')
clinical = load_processed('clinical_cleaned')

## A. Tumor vs Normal ERBB2 Expression

In [ ]:
erbb2 = rna[['sample_type', 'ERBB2']].copy() if 'ERBB2' in rna.columns else None

if erbb2 is not None:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=erbb2, x='sample_type', y='ERBB2', palette='Set2', ax=ax)
    ax.set_title('ERBB2 Expression by Sample Type')
    ax.set_ylabel('ERBB2 (log2 CPM)')
    ax.set_xlabel('')
    plt.xticks(rotation=20)
    save_fig('05_erbb2_sample_type')
    plt.show()

## B. Differential Expression: HER2+ vs HER2−

Simple t-test per gene; use adjusted p-values (Benjamini-Hochberg) for volcano plot.

In [ ]:
from statsmodels.stats.multitest import multipletests

tumor = rna[rna['sample_type'] == 'Primary Tumor']
gene_cols = [c for c in tumor.columns if c != 'sample_type']

her2_pos_idx = clinical[clinical['her2_positive'] == True].index
her2_neg_idx = clinical[clinical['her2_positive'] == False].index

pos = tumor.reindex(her2_pos_idx)[gene_cols].dropna(how='all')
neg = tumor.reindex(her2_neg_idx)[gene_cols].dropna(how='all')

print(f'HER2+ samples: {len(pos)}, HER2− samples: {len(neg)}')

In [ ]:
# Run t-tests
pvals, logfc = [], []
for gene in gene_cols:
    g1 = pos[gene].dropna()
    g2 = neg[gene].dropna()
    if len(g1) < 3 or len(g2) < 3:
        pvals.append(1.0)
        logfc.append(0.0)
        continue
    t, p = stats.ttest_ind(g1, g2)
    pvals.append(p)
    logfc.append(g1.mean() - g2.mean())

de_results = pd.DataFrame({'gene': gene_cols, 'logFC': logfc, 'pval': pvals})
_, de_results['padj'], _, _ = multipletests(de_results['pval'], method='fdr_bh')
de_results = de_results.sort_values('padj')
de_results.head(20)

In [ ]:
# Volcano plot
de_results['neg_log10_padj'] = -np.log10(de_results['padj'].clip(lower=1e-300))
sig = de_results['padj'] < 0.05

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(de_results.loc[~sig, 'logFC'], de_results.loc[~sig, 'neg_log10_padj'],
           c='#BDC3C7', s=8, alpha=0.6, label='NS')
ax.scatter(de_results.loc[sig, 'logFC'], de_results.loc[sig, 'neg_log10_padj'],
           c='#E74C3C', s=10, alpha=0.8, label='FDR < 0.05')

# Label top genes
for _, row in de_results.head(10).iterrows():
    ax.text(row['logFC'], row['neg_log10_padj'], row['gene'], fontsize=7)

ax.axhline(-np.log10(0.05), color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('log2 Fold Change (HER2+ vs HER2−)')
ax.set_ylabel('−log10 adjusted p-value')
ax.set_title('Differential Expression: HER2+ vs HER2−')
ax.legend()
save_fig('05_volcano')
plt.show()

## C. Clinical Correlates of HER2 Positivity

TODO: explore age, stage, grade, ER/PR status

In [ ]:
clinical.groupby('her2_positive').describe()

## D. Triple-Negative Breast Cancer (TNBC) Subtyping

TODO: identify ER−/PR−/HER2− patients and characterize their expression profile

In [ ]:
# TODO: requires ER and PR status columns from clinical data
# tnbc = clinical[(clinical['er_status'] == 'Negative') &
#                 (clinical['pr_status'] == 'Negative') &
#                 (clinical['her2_positive'] == False)]